In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
from tqdm import tqdm

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from spike_classifier import prepare_data, annotate_spikes, train_classifier

# Use the ROI data that already has labeled ROIs
ROI_DATA_DIR = Path(r"C:\Users\mzinn1\Desktop\val_spike_data")
ROI_DATA_PATH = ROI_DATA_DIR / "all_roi_features.npy"

# Spike keys CSV will be created alongside the ROI data
SPIKE_KEYS_CSV = ROI_DATA_PATH.parent / f"{ROI_DATA_PATH.stem}_spike_keys.csv"

MODEL_OUT_DIR = Path(r"C:\Users\mzinn1\Desktop\gcamp_model")
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

ROI_DATA_PATH, SPIKE_KEYS_CSV, MODEL_OUT_DIR

(WindowsPath('C:/Users/mzinn1/Desktop/val_spike_data/all_roi_features.npy'),
 WindowsPath('C:/Users/mzinn1/Desktop/val_spike_data/all_roi_features_spike_keys.csv'),
 WindowsPath('C:/Users/mzinn1/Desktop/gcamp_model'))

In [2]:
def summarize_spike_data(npy_path: Path, csv_path: Path):
    """Summarize spike data from ROI .npy and spike keys CSV."""
    if not npy_path.exists():
        print(f"Missing: {npy_path}")
        return
    
    d = np.load(npy_path, allow_pickle=True).item()
    
    # Count ROIs with spikes
    rois_with_spikes = sum(1 for roi in d.values() if roi.get('spikes'))
    total_spikes = sum(len(roi.get('spikes', {})) for roi in d.values())
    
    print(f"ROI Data: {npy_path}")
    print(f"Total ROIs: {len(d)}")
    print(f"ROIs with spikes: {rois_with_spikes}")
    print(f"Total spikes: {total_spikes}")
    
    # Check CSV for labels
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        n_good = (df['label'] == 1).sum()
        n_bad = (df['label'] == 0).sum()
        n_unlabeled = (df['label'] == -1).sum()
        
        print(f"\nSpike Labels (from CSV):")
        print(f"  Good: {n_good} | Bad: {n_bad} | Unlabeled: {n_unlabeled}")
    else:
        print(f"\nSpike keys CSV not found: {csv_path}")
        print("Run prepare_data to generate spike features first.")

summarize_spike_data(ROI_DATA_PATH, SPIKE_KEYS_CSV)

ROI Data: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy
Total ROIs: 279
ROIs with spikes: 126
Total spikes: 2245

Spike Labels (from CSV):
  Good: 247 | Bad: 855 | Unlabeled: 1143


In [9]:
def reset_spike_labels(csv_path: Path, npy_path: Path):
    """Reset all spike labels to -1 (unlabeled)."""
    if not csv_path.exists():
        print(f"❌ CSV not found: {csv_path}")
        return
    
    # Reset CSV
    df = pd.read_csv(csv_path)
    n_labeled = (df['label'] != -1).sum()
    df['label'] = -1
    df.to_csv(csv_path, index=False)
    
    # Reset .npy file
    if npy_path.exists():
        npy_dict = np.load(npy_path, allow_pickle=True).item()
        for roi_key, roi_data in npy_dict.items():
            if 'spikes' in roi_data:
                for spike_idx in roi_data['spikes']:
                    roi_data['spikes'][spike_idx]['label'] = -1
        np.save(npy_path, npy_dict, allow_pickle=True)
    
    print(f"✅ Reset {n_labeled} labels to -1 (unlabeled)")
    print(f"   CSV: {csv_path}")
    print(f"   NPY: {npy_path}")

reset_spike_labels(SPIKE_KEYS_CSV, ROI_DATA_PATH)
summarize_spike_data(ROI_DATA_PATH, SPIKE_KEYS_CSV)

✅ Reset 1163 labels to -1 (unlabeled)
   CSV: C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features_spike_keys.csv
   NPY: C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
ROI Data: C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
Total ROIs: 16828
ROIs with spikes: 419
Total spikes: 11352

Spike Labels (from CSV):
  Good: 0 | Bad: 0 | Unlabeled: 11352


In [3]:



roi_dict = prepare_data.main(
    input_path=str(ROI_DATA_PATH),
    output_path=None,  
    max_rois=None      
)

summarize_spike_data(ROI_DATA_PATH, SPIKE_KEYS_CSV)

Processed 126 good ROIs, found 2245 spikes
Skipped 153 bad ROIs
ROI Data: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy
Total ROIs: 279
ROIs with spikes: 126
Total spikes: 2245

Spike Labels (from CSV):
  Good: 247 | Bad: 855 | Unlabeled: 1143


In [9]:
from spike_classifier.annotate_spikes import annotate_spikes_by_roi as annotate_spikes

N_ANNOTATIONS = 5000

annotate_spikes(
    data_path=ROI_DATA_PATH,
    unlabeled_only=True
)

Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features_spike_keys.csv
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features_spike_keys.csv
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features_spike_keys.csv
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features_spike_keys.csv
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features_spike_keys.csv
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features_spike_keys.csv
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy
Saved: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features_

SessionStats(rois_total=126, rois_done=72, spikes_total_in_session=0, spikes_labeled=1140, spikes_updated=1136, spikes_confirmed=4, spikes_skipped=0)

In [10]:
# Cell 5: Check Label Distribution
summarize_spike_data(ROI_DATA_PATH, SPIKE_KEYS_CSV)

# Check class balance
df = pd.read_csv(SPIKE_KEYS_CSV)
good = (df['label'] == 1).sum()
bad = (df['label'] == 0).sum()
print(f"\nGood/Bad ratio: {good} / {bad}")

if good < 10 or bad < 10:
    print("⚠️ Warning: Need more labeled samples for training!")

ROI Data: C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy
Total ROIs: 279
ROIs with spikes: 126
Total spikes: 2245

Spike Labels (from CSV):
  Good: 247 | Bad: 855 | Unlabeled: 1143

Good/Bad ratio: 247 / 855


In [4]:
from spike_classifier.train_classifier import train_spike_classifier

results = train_spike_classifier(
    data_path=ROI_DATA_PATH,
    output_dir=MODEL_OUT_DIR
)

best_model = results['best_model']
config = results['config']
results_df = results['results_df']

print(f"\n🏆 Best model: {config['model_type']} with {config['transform']} transform")
print(f"Test accuracy: {config['test_accuracy']:.4f}")
print(f"ROC AUC: {config['roc_auc']:.4f}")
print(f"Selected features: {config['selected_features']}")

Loading spike data from C:\Users\mzinn1\Desktop\val_spike_data\all_roi_features.npy...
Total spikes: 2245
Labeled spikes: 1102
  - Good spikes (label=1): 247
  - Bad spikes (label=0): 855

Successfully extracted features for 1102 labeled spikes
Feature matrix shape: (1102, 4)

Dataset Summary
Total labeled spikes: 1102
Number of features: 4
Feature names: ['distance', 'dominance_score', 'mini_prom', 'spike_prom']
Label distribution:
  - Bad (0):  855
  - Good (1): 247

Train set: 881 samples
  - Bad (0):  684
  - Good (1): 197
Test set: 221 samples
  - Bad (0):  171
  - Good (1): 50

TESTING RANDOM FOREST
  RAW   : CV Acc: 0.9830 (+/- 0.0124) | Test Acc: 0.9683 | ROC AUC: 0.9935
  ✓ New best!
  LOG   : CV Acc: 0.9750 (+/- 0.0301) | Test Acc: 0.9638 | ROC AUC: 0.9828
  SQRT  : CV Acc: 0.9716 (+/- 0.0366) | Test Acc: 0.9638 | ROC AUC: 0.9823
  SQUARE: CV Acc: 0.9796 (+/- 0.0310) | Test Acc: 0.9729 | ROC AUC: 0.9847
  ✓ New best!

TESTING LOGISTIC REGRESSION
  RAW   : CV Acc: 0.9637 (+/- 